# Cambodia Tourism Chatbot — Prediction and Error Analysis

This notebook loads the trained SimpleRNN model from the training notebook and evaluates its behavior on a range of test inputs. The goal is not only to demonstrate that the chatbot produces responses, but also to systematically identify and explain the situations in which it fails.

The notebook is organized as follows:

1. Load the saved model, tokenizer, and configuration
2. Define a response generation function
3. Test the chatbot on a variety of in-domain questions
4. Probe the chatbot's failure modes with carefully chosen edge cases
5. Summarize the findings in a results table
6. Discuss why these failures occur

## 1. Load the Trained Model and Artifacts

We load the three files saved by `train.ipynb`: the Keras model (`model.h5`), the fitted tokenizer (`tokenizer.pkl`), and the configuration dictionary (`config.pkl`) which stores the maximum sequence length used during training. We also build a reverse mapping from integer indices back to words, which is needed to convert model predictions into readable text.

In [2]:
import numpy as np
import pickle
import pandas as pd
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

model = load_model("model.h5")

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

with open("config.pkl", "rb") as f:
    config = pickle.load(f)

max_len = config["max_len"]
vocab_size = config["vocab_size"]
index_to_word = {v: k for k, v in tokenizer.word_index.items()}

print(f"Model loaded.")
print(f"Vocabulary size: {vocab_size}")
print(f"Max sequence length: {max_len}")

Model loaded.
Vocabulary size: 422
Max sequence length: 12


## 2. Define the Response Generation Function

Given an input question, the function performs the following steps:

1. Lowercase the input and convert it to a sequence of integer tokens.
2. Pad or truncate the sequence to `max_len`.
3. Run the model forward pass to obtain a probability distribution over the vocabulary at each output position.
4. Apply greedy decoding: take the token with the highest probability at each position (`argmax`).
5. Convert the predicted indices back to words, dropping padding (`0`) and unknown (`<OOV>`) tokens.

Greedy decoding is the simplest decoding strategy. More advanced strategies such as beam search or sampling could produce more diverse responses, but greedy decoding is sufficient to demonstrate the model's behavior.

In [3]:
def generate_response(text):
    seq = tokenizer.texts_to_sequences([text.lower()])
    seq = pad_sequences(seq, maxlen=max_len, padding='post')

    pred = model.predict(seq, verbose=0)
    pred_ids = np.argmax(pred, axis=-1)[0]

    words = []
    for idx in pred_ids:
        if idx == 0:
            continue
        word = index_to_word.get(idx, '')
        if word and word != '<OOV>':
            words.append(word)

    return " ".join(words) if words else "(no response)"

print("generate_response() defined")

generate_response() defined


## 3. Test on In-Domain Questions

We first evaluate the chatbot on questions that are similar in style to the training data. These are the questions the model is most likely to answer well. The goal is to establish a baseline for what "working" looks like before we deliberately stress-test the system.

In [4]:
in_domain_questions = [
    "Where is Angkor Wat?",
    "What is the capital of Cambodia?",
    "What food should I try in Cambodia?",
    "What currency is used in Cambodia?",
    "When is the best time to visit Cambodia?",
]

for q in in_domain_questions:
    print(f"Q: {q}")
    print(f"A: {generate_response(q)}")
    print("-" * 60)

Q: Where is Angkor Wat?
A: it is is and and and and and and and and and
------------------------------------------------------------
Q: What is the capital of Cambodia?
A: it is gmt plus the
------------------------------------------------------------
Q: What food should I try in Cambodia?
A: it should try amok and lok lak lak lak lak lak lak
------------------------------------------------------------
Q: What currency is used in Cambodia?
A: it is is riel local
------------------------------------------------------------
Q: When is the best time to visit Cambodia?
A: it is usually sunrise is the best
------------------------------------------------------------


## 4. Probe Failure Modes

To understand the chatbot's limitations, we now test categories of inputs that are known to be challenging for vanilla recurrent neural networks. Each category targets a different weakness of the SimpleRNN architecture.

### 4.1 Out-of-Vocabulary Words

The tokenizer assigns the `<OOV>` token to any word it has not seen during training. When the model receives an `<OOV>` token, it has no learned embedding to interpret it, so the response cannot meaningfully depend on the unknown word.

In [5]:
oov_questions = [
    "Tell me about cryptocurrency in Cambodia",
    "What about quantum computing tourism?",
    "Can I find vegan keto restaurants?",
]

for q in oov_questions:
    print(f"Q: {q}")
    print(f"A: {generate_response(q)}")
    print("-" * 60)

Q: Tell me about cryptocurrency in Cambodia
A: it is is the but
------------------------------------------------------------
Q: What about quantum computing tourism?
A: it is grab a daily daily daily daily daily daily daily daily
------------------------------------------------------------
Q: Can I find vegan keto restaurants?
A: it should khan is english watch watch watch watch watch watch watch
------------------------------------------------------------


### 4.2 Paraphrasing

We ask the same underlying question in different surface forms. A robust language model should give similar answers for paraphrases. SimpleRNN, with no semantic understanding beyond surface co-occurrence statistics, is expected to be sensitive to wording.

In [6]:
paraphrases = [
    "What food can I eat in Cambodia?",
    "What dishes are popular in Cambodia?",
    "Tell me about Cambodian cuisine",
    "What should I eat when visiting Cambodia?",
]

for q in paraphrases:
    print(f"Q: {q}")
    print(f"A: {generate_response(q)}")
    print("-" * 60)

Q: What food can I eat in Cambodia?
A: it should cover and a a
------------------------------------------------------------
Q: What dishes are popular in Cambodia?
A: it is try is chum or or or or or or or
------------------------------------------------------------
Q: Tell me about Cambodian cuisine
A: it is is the but but but but but but but but
------------------------------------------------------------
Q: What should I eat when visiting Cambodia?
A: it should and and seven
------------------------------------------------------------


### 4.3 Long Inputs

SimpleRNN suffers from vanishing gradients, which means information from the beginning of a long sequence is effectively lost by the time the network reaches the end. We test this by feeding very long questions and observing whether the model can still extract the relevant intent.

In [7]:
long_questions = [
    "I am planning a trip to Cambodia next month with my family and we have never been to Southeast Asia before, so could you tell me what the most important things to know about Cambodian culture are?",
    "My friends and I are backpackers visiting many countries in Asia and we want to spend about two weeks in Cambodia, where should we go?",
]

for q in long_questions:
    print(f"Q: {q}")
    print(f"A: {generate_response(q)}")
    print("-" * 60)

Q: I am planning a trip to Cambodia next month with my family and we have never been to Southeast Asia before, so could you tell me what the most important things to know about Cambodian culture are?
A: of the pepper
------------------------------------------------------------
Q: My friends and I are backpackers visiting many countries in Asia and we want to spend about two weeks in Cambodia, where should we go?
A: most can about a speed ferry siem trees penh
------------------------------------------------------------


### 4.4 Very Short Inputs

At the opposite extreme, single-word or two-word inputs provide very little context for the model. After the input tokens, the rest of the sequence is padding, and the model must produce an answer with almost no signal to condition on.

In [8]:
short_questions = [
    "food",
    "hello",
    "Cambodia?",
]

for q in short_questions:
    print(f"Q: {q}")
    print(f"A: {generate_response(q)}")
    print("-" * 60)

Q: food
A: lok lok lok lok lok lok lok lok lok lok lok lok
------------------------------------------------------------
Q: hello
A: tourist tourist tourist tourist tourist tourist tourist tourist tourist tourist tourist tourist
------------------------------------------------------------
Q: Cambodia?
A: widely widely widely widely widely widely widely widely widely widely widely widely
------------------------------------------------------------


### 4.5 Off-Topic Inputs

The training data is restricted to Cambodia tourism. When asked about unrelated topics, the model has no relevant learned patterns and is forced to produce something from its limited vocabulary. We expect generic, repetitive, or topically incorrect responses.

In [14]:
off_topic_questions = [
    "What is the meaning of life?",
    "How do I bake a chocolate cake?",
    "Who won the World Cup last year?",
    "What is the AI?",
]

for q in off_topic_questions:
    print(f"Q: {q}")
    print(f"A: {generate_response(q)}")
    print("-" * 60)

Q: What is the meaning of life?
A: it is gmt plus the the the the the the the the
------------------------------------------------------------
Q: How do I bake a chocolate cake?
A: passapp can take a visa night or or or or or or
------------------------------------------------------------
Q: Who won the World Cup last year?
A: most is better on most with
------------------------------------------------------------
Q: What is the AI?
A: it is gmt plus plus plus plus plus plus plus plus plus
------------------------------------------------------------


## 5. Summary Table

We collect the responses across all test categories into a single table. This makes it easy to compare behavior side by side and serves as evidence for the discussion in the report.

In [10]:
test_set = [
    ("In-domain",   "Where is Angkor Wat?"),
    ("In-domain",   "What is the capital of Cambodia?"),
    ("In-domain",   "What currency is used in Cambodia?"),
    ("OOV words",   "Tell me about cryptocurrency in Cambodia"),
    ("Paraphrase",  "What dishes are popular in Cambodia?"),
    ("Paraphrase",  "Tell me about Cambodian cuisine"),
    ("Long input",  "I am planning a trip to Cambodia next month with my family and we have never been to Southeast Asia before, so could you tell me what the most important things to know about Cambodian culture are?"),
    ("Short input", "food"),
    ("Off-topic",   "What is the meaning of life?"),
    ("Off-topic",   "How do I bake a chocolate cake?"),
]

rows = []
for category, q in test_set:
    a = generate_response(q)
    rows.append({"Category": category, "Question": q, "Response": a})

results_df = pd.DataFrame(rows)
results_df

,Category,Question,Response
0,In-domain,Where is Angkor Wat?,it is is and and and and and and and and and
1,In-domain,What is the capital of Cambodia?,it is gmt plus the
2,In-domain,What currency is used in Cambodia?,it is is riel local
3,OOV words,Tell me about cryptocurrency in Cambodia,it is is the but
4,Paraphrase,What dishes are popular in Cambodia?,it is try is chum or or or or or or or
5,Paraphrase,Tell me about Cambodian cuisine,it is is the but but but but but but but but
6,Long input,I am planning a trip to Cambodia next month wi...,of the pepper
7,Short input,food,lok lok lok lok lok lok lok lok lok lok lok lok
8,Off-topic,What is the meaning of life?,it is gmt plus the the the the the the the the
9,Off-topic,How do I bake a chocolate cake?,passapp can take a visa night or or or or or or
